# Capacitance matrix and LOM analysis
### Prerequisite
You need to have a working local installation of Ansys.

## 1. Create the design in Metal

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, Headings

In [3]:
design = designs.DesignPlanar()
gui = MetalGUI(design)

from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander

In [4]:
design.variables['cpw_width'] = '15 um'
design.variables['cpw_gap'] = '9 um'

### In this example, the design consists of 4 qubits and 4 CPWs

In [5]:
# Allow running the same cell here multiple times to overwrite changes
design.overwrite_enabled = True

## Custom options for all the transmons
options = dict(
    # Some options we want to modify from the defaults
    # (see below for defaults)
    pad_width = '425 um', 
    pocket_height = '650um',
    # Adding 4 connectors (see below for defaults)
    connection_pads=dict(
        readout = dict(loc_W=+1,loc_H=-1, pad_width='200um'),
        bus1 = dict(loc_W=-1,loc_H=+1, pad_height='30um'),
        bus2 = dict(loc_W=-1,loc_H=-1, pad_height='50um')
    )
)

## Create 4 transmons

q1 = TransmonPocket(design, 'Q1', options = dict(
    pos_x='+2.42251mm', pos_y='+0.0mm', **options))
q2 = TransmonPocket(design, 'Q2', options = dict(
    pos_x='+0.0mm', pos_y='-0.95mm', orientation = '270', **options))
q3 = TransmonPocket(design, 'Q3', options = dict(
    pos_x='-2.42251mm', pos_y='+0.0mm', orientation = '180', **options))
q4 = TransmonPocket(design, 'Q4', options = dict(
    pos_x='+0.0mm', pos_y='+0.95mm', orientation = '90', **options))

RouteMeander.get_template_options(design)

options = Dict(
        lead=Dict(
            start_straight='0.2mm',
            end_straight='0.2mm'),
        trace_gap='9um',
        trace_width='15um')

def connect(component_name: str, component1: str, pin1: str, component2: str, pin2: str,
            length: str, asymmetry='0 um', flip=False, fillet='90um'):
    """Connect two pins with a CPW."""
    myoptions = Dict(
        fillet=fillet,
        hfss_wire_bonds = True,
        pin_inputs=Dict(
            start_pin=Dict(
                component=component1,
                pin=pin1),
            end_pin=Dict(
                component=component2,
                pin=pin2)),
        total_length=length)
    myoptions.update(options)
    myoptions.meander.asymmetry = asymmetry
    myoptions.meander.lead_direction_inverted = 'true' if flip else 'false'
    return RouteMeander(design, component_name, myoptions)

asym = 140
cpw1 = connect('cpw1', 'Q1', 'bus2', 'Q2', 'bus1', '6.0 mm', f'+{asym}um')
cpw2 = connect('cpw2', 'Q3', 'bus1', 'Q2', 'bus2', '6.1 mm', f'-{asym}um', flip=True)
cpw3 = connect('cpw3', 'Q3', 'bus2', 'Q4', 'bus1', '6.0 mm', f'+{asym}um')
cpw4 = connect('cpw4', 'Q1', 'bus1', 'Q4', 'bus2', '6.1 mm', f'-{asym}um', flip=True)

gui.rebuild()
gui.autoscale()

## 2. Capacitance Analysis and LOM derivation using the analysis package - most users

### Capacitance Analysis
Select the analysis you intend to run from the `qiskit_metal.analyses` collection.<br>
Select the design to analyze and the tool to use for any external simulation

In [6]:
from qiskit_metal.analyses.quantization import LOManalysis
c1 = LOManalysis(design, "q3d")

(optional) You can review and update the Analysis default setup following the examples in the next two cells.

In [7]:
c1.sim.setup

{'name': 'Setup',
 'reuse_selected_design': True,
 'reuse_setup': True,
 'freq_ghz': 5.0,
 'save_fields': False,
 'enabled': True,
 'max_passes': 15,
 'min_passes': 2,
 'min_converged_passes': 2,
 'percent_error': 0.5,
 'percent_refinement': 30,
 'auto_increase_solution_order': True,
 'solution_order': 'High',
 'solver_type': 'Iterative'}

In [8]:
# example: update single setting
c1.sim.setup.max_passes = 6
# example: update multiple settings
c1.sim.setup_update(solution_order = 'Medium', auto_increase_solution_order = 'False')

c1.sim.setup

{'name': 'Setup',
 'reuse_selected_design': True,
 'reuse_setup': True,
 'freq_ghz': 5.0,
 'save_fields': False,
 'enabled': True,
 'max_passes': 6,
 'min_passes': 2,
 'min_converged_passes': 2,
 'percent_error': 0.5,
 'percent_refinement': 30,
 'auto_increase_solution_order': 'False',
 'solution_order': 'Medium',
 'solver_type': 'Iterative'}

Analyze a single qubit with 2 endcaps using the default (or edited) analysis setup. Then show the capacitance matrix (from the last pass).

You can use the method `run()` instead of `sim.run()` in the following cell if you want to run both cap extraction and lom analysis in a single step. If so, make sure to also tweak the setup for the lom analysis. The input parameters are otherwise the same for the two methods. 

In [10]:
c1.sim.run(components=['Q1'], open_terminations=[('Q1', 'readout'), ('Q1', 'bus1'), ('Q1', 'bus2')])
c1.sim.capacitance_matrix

INFO 03:57PM [connect_project]: Connecting to Ansys Desktop API...
INFO 03:57PM [load_ansys_project]: 	Opened Ansys App
INFO 03:57PM [load_ansys_project]: 	Opened Ansys Desktop v2024.2.0
INFO 03:57PM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/jaseung/Documents/Ansoft/
	Project:   Project5
INFO 03:57PM [connect_design]: No active design found (or error getting active design).
INFO 03:57PM [connect]: 	 Connected to project "Project5". No design detected
INFO 03:59PM [connect_design]: 	Opened active design
	Design:    Design_q3d [Solution type: Q3D]
WARNING 03:59PM [connect_setup]: 	No design setup detected.
WARNING 03:59PM [connect_setup]: 	Creating Q3D default setup.
INFO 03:59PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 03:59PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 03:59PM [analyze]: Analyzing setup Setup
INFO 04:00PM [get_matrix]: Exporting matrix data to (C:\Users\jaseung\AppData\Local

,bus1_connector_pad_Q1,bus2_connector_pad_Q1,ground_main_plane,pad_bot_Q1,pad_top_Q1,readout_connector_pad_Q1
bus1_connector_pad_Q1,49.81917,-0.42243,-33.47371,-1.56619,-13.23557,-0.20510
bus2_connector_pad_Q1,-0.42243,53.94438,-35.81375,-13.86735,-1.83231,-1.01548
ground_main_plane,-33.47371,-35.81375,237.80297,-31.52796,-37.95933,-36.57183
pad_bot_Q1,-1.56619,-13.86735,-31.52796,98.12448,-30.21163,-18.79337
pad_top_Q1,-13.23557,-1.83231,-37.95933,-30.21163,88.15054,-2.21079
readout_connector_pad_Q1,-0.20510,-1.01548,-36.57183,-18.79337,-2.21079,59.86763


(otional - case-dependent)<br>If the previous cell was interrupted due to license limitations and for any reason you finally manually launched the simulation from the renderer GUI (outside qiskit-metal) you might be able to recover the simulation results by uncommenting and executing the following cell

In [ ]:
#c1.sim._get_results_from_renderer()
#c1.sim.capacitance_matrix

The last variables you pass to the `run()` or `sim.run()` methods, will be stored in the `sim.setup` dictionary under the key `run`. You can recall the information passed by either accessing the dictionary directly, or by using the print handle below.

In [ ]:
# c1.setup.run    <- direct access
c1.sim.print_run_args()

You can re-run the analysis after varying the parameters.<br>
Not passing the parameter `components` to the `sim.run()` method, skips the rendering and tries to run the analysis on the latest design. If a design is not found, the full metal design is rendered.

In [11]:
c1.sim.setup.freq_ghz = 4.8
c1.sim.run()
c1.sim.capacitance_matrix

INFO 04:00PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 04:00PM [analyze]: Analyzing setup Setup
INFO 04:01PM [get_matrix]: Exporting matrix data to (C:\Users\jaseung\AppData\Local\Temp\tmp35d34qbw.txt, C, , Setup:LastAdaptive, "Original", "ohm", "nH", "fF", "mSie", 4800000000, Maxwell, 1, False
INFO 04:01PM [get_matrix]: Exporting matrix data to (C:\Users\jaseung\AppData\Local\Temp\tmpea4ha12u.txt, C, , Setup:AdaptivePass, "Original", "ohm", "nH", "fF", "mSie", 4800000000, Maxwell, 1, False
INFO 04:01PM [get_matrix]: Exporting matrix data to (C:\Users\jaseung\AppData\Local\Temp\tmpzeeilg7z.txt, C, , Setup:AdaptivePass, "Original", "ohm", "nH", "fF", "mSie", 4800000000, Maxwell, 2, False
INFO 04:01PM [get_matrix]: Exporting matrix data to (C:\Users\jaseung\AppData\Local\Temp\tmptle6ahqi.txt, C, , Setup:AdaptivePass, "Original", "ohm", "nH", "fF", "mSie", 4800000000, Maxwell, 3, False
INFO 04:01PM [get_matrix]: Exporting matrix data to (C:\Users\jaseu

,bus1_connector_pad_Q1,bus2_connector_pad_Q1,ground_main_plane,pad_bot_Q1,pad_top_Q1,readout_connector_pad_Q1
bus1_connector_pad_Q1,49.81917,-0.42243,-33.47371,-1.56619,-13.23557,-0.20510
bus2_connector_pad_Q1,-0.42243,53.94438,-35.81375,-13.86735,-1.83231,-1.01548
ground_main_plane,-33.47371,-35.81375,237.80297,-31.52796,-37.95933,-36.57183
pad_bot_Q1,-1.56619,-13.86735,-31.52796,98.12448,-30.21163,-18.79337
pad_top_Q1,-13.23557,-1.83231,-37.95933,-30.21163,88.15054,-2.21079
readout_connector_pad_Q1,-0.20510,-1.01548,-36.57183,-18.79337,-2.21079,59.86763


In [12]:
type(c1.sim.capacitance_matrix)

pandas.core.frame.DataFrame

### Lumped oscillator model (LOM)

Using capacitance matrices obtained from each pass, save the many parameters of the Hamiltonian of the system. `get_lumped_oscillator()` operates on 4 setup parameters: <br><br>
Lj: float <br>
Cj: float <br>
fr: Union[list, float] <br>
fb: Union[list, float] <br>

In [13]:
c1.setup.junctions = Dict({'Lj': 12.31, 'Cj': 2})
c1.setup.freq_readout = 7.0
c1.setup.freq_bus = [6.0, 6.2]

c1.run_lom()
c1.lumped_oscillator_all

[3, 4] [5 0 1]
Predicted Values

Transmon Properties
f_Q 5.419121 [GHz]
EC 311.261765 [MHz]
EJ 13.273404 [GHz]
alpha -362.874787 [MHz]
dispersion 45.611668 [KHz]
Lq 12.305036 [nH]
Cq 62.231312 [fF]
T1 35.937842 [us]

**Coupling Properties**

tCqbus1 7.378434 [fF]
gbus1_in_MHz 113.979998 [MHz]
χ_bus1 -3.131317 [MHz]
1/T1bus1 2771.367689 [Hz]
T1bus1 57.428303 [us]

tCqbus2 -6.477985 [fF]
gbus2_in_MHz -85.977626 [MHz]
χ_bus2 -9.828680 [MHz]
1/T1bus2 1184.140977 [Hz]
T1bus2 134.405401 [us]

tCqbus3 5.335202 [fF]
gbus3_in_MHz 73.127817 [MHz]
χ_bus3 -4.375136 [MHz]
1/T1bus3 473.108512 [Hz]
T1bus3 336.402620 [us]
Bus-Bus Couplings
gbus1_2 7.114556 [MHz]
gbus1_3 9.888704 [MHz]
gbus2_3 5.359322 [MHz]


,fQ,EC,EJ,alpha,dispersion,gbus,chi_in_MHz,χr MHz,gr MHz
1,5.748331,353.260691,13.273404,-417.414611,135.303742,"[108.86761985266924, -73.17555640481613, 76.49...","[-4.799113286754579, -26.58078898004884, -12.4...",4.799113,108.867620
2,5.647166,340.02625,13.273404,-400.083713,98.212636,"[112.02489141691468, -82.73776688714584, 68.84...","[-4.299348228750533, -20.660951854335774, -7.2...",4.299348,112.024891
3,5.557843,328.584288,13.273404,-385.208019,73.285465,"[110.75067006603047, -83.82114945286732, 70.94...","[-3.647572107050774, -14.838438186356596, -5.9...",3.647572,110.750670
4,5.492178,320.317004,13.273404,-374.521304,58.729258,"[110.1105756148939, -84.06701526377239, 71.178...","[-3.259741148063827, -11.855749782620117, -4.9...",3.259741,110.110576
5,5.44816,314.843098,13.273404,-367.473653,50.474219,"[112.62962174466026, -84.5569764549961, 71.592...","[-3.1921692887448785, -10.39946713142771, -4.5...",3.192169,112.629622
6,5.419121,311.261765,13.273404,-362.874787,45.611668,"[113.97999777608511, -85.97762612571542, 73.12...","[-3.131316725595434, -9.82868011560709, -4.375...",3.131317,113.979998


In [14]:
c1.plot_convergence();
c1.plot_convergence_chi()

  self._hfss_variables[variation] = pd.Series(

INFO 04:02PM [hfss_report_full_convergence]: Creating report for variation 0


Design "Design_q3d" info:
	# eigenmodes    0
	# variations    1


Once you are done with your analysis, please close it with `close()`. This will free up resources currently occupied by qiskit-metal to communiate with the tool.

In [ ]:
c1.sim.close()

: 

## 3. Directly access the renderer to modify other parameters

In [ ]:
c1.sim.start()
c1.sim.renderer

Every renderer will have its own collection of methods. Below an example with q3d

##### Prepare and run a collection of predefined setups

This is equivalent to going to the Project Manager panel in Ansys, right clicking on Analysis within the active Q3D design, selecting "Add Solution Setup...", and choosing/entering default values in the resulting popup window. You might want to do this to keep track of different solution setups, giving each of them a different/specific name.

In [ ]:
setup = c1.sim.renderer.new_ansys_setup(name = "Setup_demo", max_passes = 6)

You can directly pass to `new_ansys_setup` all the setup parameters. Of course you will then need to run the individual setups by name as well.

In [ ]:
c1.sim.renderer.analyze_setup(setup.name)

##### Get the capactiance matrix at a different pass

You might want to use this if you intend to know what was the matrix at a different pass of the simulation.

In [ ]:
# Using the analysis results, get its capacitance matrix as a dataframe.
c1.sim.renderer.get_capacitance_matrix(variation = '', solution_kind = 'AdaptivePass', pass_number = 5)

### Code to swap rows and columns in capacitance matrix
from qiskit_metal.analyses.quantization.lumped_capacitive import df_reorder_matrix_basis

df_reorder_matrix_basis(fourq_q3d.get_capacitance_matrix(), 1, 2)

##### Close the renderer

In [ ]:
c1.sim.close()